# **ColabProSST**

ColabProSST brings the ProSST backbone into the SaprotHub workflow with a compact SaProt-style entry point. ProSST is not SaProt: it uses amino-acid `input_ids` together with official ProSST `ss_input_ids`, so every task needs either `structure_tokens`, `pdb_path`/`structure_path`, or structure tokens generated in this notebook.

Supported in this first version: structure-token conversion, zero-shot mutation effect prediction, protein-level classification fine-tuning, protein-level regression fine-tuning, checkpoint prediction, and optional Hugging Face upload.


In [ ]:
#@title **Click the run-button to prepare ColabProSST**
import os
import shutil
import subprocess
import sys
from pathlib import Path

SAPROTHUB_REPO = 'https://github.com/Hello-github-code/SaprotHub' #@param {type:'string'}
SAPROTHUB_BRANCH = 'prosst' #@param {type:'string'}
DOWNLOAD_CSV_TEMPLATES = False #@param {type:'boolean'}

ROOT = Path('/content')
SAPROT_HOME = ROOT / 'SaprotHub'
PROSST_HOME = ROOT / 'ProSST'
PYG_HOME = ROOT / '.cache' / 'pyg'
HF_HOME = ROOT / '.cache' / 'huggingface'
for path in [PYG_HOME, HF_HOME, ROOT / 'prosst_uploads', ROOT / 'prosst_structure_assets']:
    path.mkdir(parents=True, exist_ok=True)
os.environ['PYG_HOME'] = str(PYG_HOME)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def add_project_paths():
    os.environ['PROSST_HOME'] = str(PROSST_HOME)
    for p in [str(SAPROT_HOME), str(SAPROT_HOME / 'saprot'), str(PROSST_HOME)]:
        if p not in sys.path:
            sys.path.insert(0, p)

def load_colabprosst_workflow():
    add_project_paths()
    import saprot  # noqa: F401
    import numpy as np  # noqa: F401
    import pandas as pd  # noqa: F401
    import torch  # noqa: F401
    import torch_geometric  # noqa: F401
    import torch_scatter  # noqa: F401
    from prosst.structure.get_sst_seq import SSTPredictor  # noqa: F401
    from saprot.utils.colab_prosst_workflow import ColabProSSTWorkflow

    return ColabProSSTWorkflow

def run_checked(command):
    status = os.system(command)
    if status != 0:
        raise RuntimeError(f'Command failed with exit status {status}: {command}')

try:
    ColabProSSTWorkflow = load_colabprosst_workflow()
except Exception:
    print("Installing ProSST...")
    if SAPROT_HOME.exists():
        shutil.rmtree(SAPROT_HOME)
    try:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', SAPROTHUB_BRANCH, SAPROTHUB_REPO, str(SAPROT_HOME)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'Could not clone {SAPROTHUB_REPO}@{SAPROTHUB_BRANCH}. '
            'Push the ColabProSST branch or set SAPROTHUB_REPO/SAPROTHUB_BRANCH '
            'to a repository that contains this notebook and the ProSST modules.'
        ) from exc
    if not PROSST_HOME.exists():
        subprocess.run(['git', 'clone', 'https://github.com/ai4protein/ProSST', str(PROSST_HOME)], check=True)

    # Keep the environment setup aligned with ColabSaprot: install SaprotHub requirements,
    # install the local package, add the widget poll helper, and remove torchao.
    run_checked(f"pip install -r {SAPROT_HOME / 'requirements.txt'}")
    run_checked(f"pip install {SAPROT_HOME}")
    run_checked("pip install jupyter_ui_poll")
    run_checked("pip uninstall -y torchao")

    for path in [
        SAPROT_HOME / 'LMDB',
        SAPROT_HOME / 'bin',
        SAPROT_HOME / 'output',
        SAPROT_HOME / 'datasets',
        SAPROT_HOME / 'adapters' / 'classification' / 'Local',
        SAPROT_HOME / 'adapters' / 'regression' / 'Local',
        SAPROT_HOME / 'adapters' / 'token_classification' / 'Local',
        SAPROT_HOME / 'adapters' / 'pair_classification' / 'Local',
        SAPROT_HOME / 'adapters' / 'pair_regression' / 'Local',
        SAPROT_HOME / 'structures',
    ]:
        path.mkdir(parents=True, exist_ok=True)
    run_checked(f"chmod +x {SAPROT_HOME / 'bin'}/*")

    # ProSST structure tokenization additionally needs PyG extensions. Install them
    # after SaprotHub pins torch, so the wheel URL matches the actual torch/CUDA pair.
    run_checked("pip install joblib==1.3.2 pathos==0.3.2 biotite==0.39.0")
    torch_info = subprocess.check_output([
        sys.executable,
        '-c',
        "import torch; print(torch.__version__.split('+')[0]); print(torch.version.cuda or 'cpu')",
    ], text=True).strip().splitlines()
    torch_version = torch_info[0]
    torch_cuda = torch_info[1]
    if torch_version.startswith('2.2.'):
        torch_version = '2.2.0'
    pyg_backend = 'cpu' if torch_cuda == 'cpu' else 'cu' + torch_cuda.replace('.', '')
    pyg_wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{pyg_backend}.html'
    print(f"Installing ProSST PyG extensions from {pyg_wheel_url}")
    run_checked(f"pip install --no-index torch_scatter torch_sparse torch_cluster torch_spline_conv -f {pyg_wheel_url}")
    run_checked("pip install torch-geometric==2.5.0")

    # Same workaround used in ColabSaprot for third-party binary library mismatch.
    keys = []
    for k in list(sys.modules.keys()):
        for sub_str in ['numpy']:
            if sub_str in k:
                keys.append(k)
    for k in keys:
        del sys.modules[k]

    ColabProSSTWorkflow = load_colabprosst_workflow()

COLABPROSST_WORKFLOW = ColabProSSTWorkflow()
COLABPROSST_WORKFLOW.create_csv_templates(download=DOWNLOAD_CSV_TEMPLATES)
print("ProSST is installed successfully!")


In [ ]:
#@title **Run ColabProSST**
from IPython.display import display

WORKFLOW = 'Convert structure to tokens' #@param ['Convert structure to tokens', 'Zero-shot mutation effect prediction', 'Fine-tune classification', 'Fine-tune regression', 'Predict classification', 'Predict regression']
INPUT_CSV = '' #@param {type:'string'}
UPLOAD_INPUT_CSV = False #@param {type:'boolean'}
STRUCTURE_FILE = '' #@param {type:'string'}
UPLOAD_STRUCTURE_FILE = False #@param {type:'boolean'}
CHAIN_ID = '' #@param {type:'string'}
USE_LAST_STRUCTURE_TOKENS = False #@param {type:'boolean'}
STRUCTURE_ZIP = '' #@param {type:'string'}
UPLOAD_STRUCTURE_ZIP = False #@param {type:'boolean'}
CHECKPOINT_PATH = '' #@param {type:'string'}
TASK_NAME = 'ProSSTUserTask' #@param {type:'string'}
NUM_LABELS = 2 #@param {type:'integer'}
MAX_EPOCHS = 2 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
FREEZE_BACKBONE = False #@param {type:'boolean'}
DOWNLOAD_OUTPUTS = True #@param {type:'boolean'}

# Advanced defaults. Most users can leave these unchanged for the first ProSST-2048 version.
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
STRUCTURE_VOCAB_SIZE = 2048 #@param {type:'integer'}
GRADIENT_CHECKPOINTING = True #@param {type:'boolean'}
OUTPUT_DIR = '/content/colabprosst_outputs' #@param {type:'string'}

if 'COLABPROSST_WORKFLOW' not in globals():
    raise RuntimeError('Run the preparation cell first.')

COLABPROSST_WORKFLOW.set_output_dir(OUTPUT_DIR)

if WORKFLOW == 'Convert structure to tokens':
    result_df = COLABPROSST_WORKFLOW.convert_structure(
        structure_path=STRUCTURE_FILE,
        upload_structure=UPLOAD_STRUCTURE_FILE,
        chain_id=CHAIN_ID,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df)

elif WORKFLOW == 'Zero-shot mutation effect prediction':
    result_df = COLABPROSST_WORKFLOW.run_zero_shot(
        input_csv=INPUT_CSV,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df.head())

elif WORKFLOW in ['Fine-tune classification', 'Fine-tune regression']:
    task_type = 'classification' if WORKFLOW.endswith('classification') else 'regression'
    result = COLABPROSST_WORKFLOW.train_downstream(
        task_type=task_type,
        input_csv=INPUT_CSV,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        task_name=TASK_NAME,
        num_labels=NUM_LABELS,
        max_epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        freeze_backbone=FREEZE_BACKBONE,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        download=DOWNLOAD_OUTPUTS,
    )
    print(result)

elif WORKFLOW in ['Predict classification', 'Predict regression']:
    task_type = 'classification' if WORKFLOW.endswith('classification') else 'regression'
    if not CHECKPOINT_PATH.strip():
        raise ValueError('Set CHECKPOINT_PATH before running prediction.')
    result_df = COLABPROSST_WORKFLOW.predict_downstream(
        task_type=task_type,
        input_csv=INPUT_CSV,
        checkpoint_path=CHECKPOINT_PATH,
        upload_csv=UPLOAD_INPUT_CSV,
        use_last_structure_tokens=USE_LAST_STRUCTURE_TOKENS,
        structure_zip=STRUCTURE_ZIP,
        upload_structure_zip=UPLOAD_STRUCTURE_ZIP,
        num_labels=NUM_LABELS,
        batch_size=BATCH_SIZE,
        model_path=MODEL_PATH,
        structure_vocab_size=STRUCTURE_VOCAB_SIZE,
        download=DOWNLOAD_OUTPUTS,
    )
    display(result_df.head())


In [ ]:
#@title **Optional: upload a trained ColabProSST checkpoint to Hugging Face**
HF_REPO_ID = '' #@param {type:'string'}
HF_PRIVATE_REPO = False #@param {type:'boolean'}
RUN_HF_LOGIN = True #@param {type:'boolean'}
CHECKPOINT_PATH = '/content/SaprotHub/weights/prosst/ProSSTUserTask.pt' #@param {type:'string'}
TASK_TYPE = 'classification' #@param ['classification', 'regression']
NUM_LABELS = 2 #@param {type:'integer'}
MODEL_PATH = 'AI4Protein/ProSST-2048' #@param {type:'string'}
MODEL_CARD_TITLE = 'ColabProSST model' #@param {type:'string'}
MODEL_DESCRIPTION = 'A ProSST checkpoint trained with ColabProSST.' #@param {type:'string'}
DOWNLOAD_MODEL_PACKAGE = False #@param {type:'boolean'}

if 'COLABPROSST_WORKFLOW' not in globals():
    raise RuntimeError('Run the preparation cell first.')

package_dir = COLABPROSST_WORKFLOW.upload_checkpoint_to_hf(
    repo_id=HF_REPO_ID,
    checkpoint_path=CHECKPOINT_PATH,
    task_type=TASK_TYPE,
    num_labels=NUM_LABELS,
    model_path=MODEL_PATH,
    private=HF_PRIVATE_REPO,
    run_login=RUN_HF_LOGIN,
    title=MODEL_CARD_TITLE,
    description=MODEL_DESCRIPTION,
    download_package=DOWNLOAD_MODEL_PACKAGE,
)
print('local package:', package_dir)
